<a href="https://colab.research.google.com/github/adhh0715/employee-knowledge-explorer-substitute-data/blob/main/Employee_Knowledge_Explorer_Substitute_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Build an AI Employee Knowledge Explorer

## Business Scenario

Northstar Systems stores employee guidance across separate HR, IT, Benefits, Finance, Onboarding, and Security documents. Employees often ask questions in their own words, which may differ from the policy wording. Northstar wants a **searchable employee knowledge explorer** that finds the most relevant information and shows its source.

You will build a guided retrieval system that:
1. Loads the supplied fictional knowledge-base documents with source and department metadata.
2. Creates a **TF-IDF + cosine similarity** baseline to represent lexical retrieval.
3. Generates **dense embeddings** and builds a **FAISS IndexFlatIP** semantic search index with stable record IDs.
4. Uses **LlamaIndex Documents, Nodes, and an IngestionPipeline** for retrieval-ready content.
5. Stores nodes and metadata in **persistent ChromaDB**, searches them, applies metadata filters, and updates a record.
6. Compares retrieval results and records the limitations of each method.

**Scope:** Google Colab; beginner–intermediate. This assignment focuses on *retrieval and context quality*, not building a web interface or training an ML model. An LLM-based answer is an optional extension.

## Learning Outcomes

By completing this assignment, you should be able to:

- Explain why context relevance, provenance, and token budget matter.
- Implement a TF-IDF baseline and use cosine similarity to rank documents.
- Generate sentence embeddings with `BAAI/bge-small-en-v1.5`.
- Build and query a normalized-vector FAISS index using `IndexFlatIP` and `IndexIDMap`.
- Convert document text and metadata into LlamaIndex `Document` and `Node` objects.
- Store/search retrieval-ready nodes in a persistent ChromaDB collection.
- Use metadata filtering, record updates, and basic retrieval evaluation.
- Identify lexical retrieval failures, cross-method differences, and unsupported questions.

## Provided Resources

Download and keep the supplied **`Northstar_Employee_Knowledge_Base.zip`** file. It contains six fictional UTF-8 text files:

| File | Department/category |
|---|---|
| `hr_leave_policy.txt` | HR |
| `it_device_support.txt` | IT |
| `benefits_and_wellbeing.txt` | Benefits |
| `finance_expense_reimbursement.txt` | Finance |
| `people_onboarding.txt` | Onboarding |
| `security_data_protection.txt` | Security |

The dataset is fictional and is for training only. Fixed test questions are provided near the end. **Do not substitute different documents** when preparing the final submission, so that results can be evaluated consistently.

## Submission

Upload your completed `.ipynb` with **executed outputs** to a GitHub repository and share the repository link. Include the six provided knowledge files or the supplied ZIP in your repository. Do **not** upload downloaded embedding-model caches, Chroma database files, API keys, or other large generated files.

## Part A — Prepare the Colab Environment and Knowledge Base
### Step 0. Install libraries and upload the supplied ZIP

Run the two setup cells below. These cells are provided so that students can focus on the 12 unfinished functions. The embedding-model download requires internet access in Colab; the required retrieval activities do **not** require an LLM API key.

- Upload the **exact supplied ZIP**, not a folder of other documents.
- The setup extracts only `.txt` files into `/content/northstar_knowledge_base`.
- Use the fixed category mapping below so search results can be compared fairly.
- If a fresh Colab runtime reports a library import error after installation, restart the runtime and rerun the setup cells.

**Purpose of Code:** Install the packages used in the three lessons, load required imports and model settings, then make the six policy documents available for later retrieval and vector-store operations.

In [1]:
!pip -q install -U scikit-learn sentence-transformers faiss-cpu chromadb llama-index-core

from pathlib import Path
import io
import zipfile
import time
import numpy as np
import pandas as pd
import faiss
import chromadb
from google.colab import files
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from llama_index.core import Document
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter

MODEL_NAME = "BAAI/bge-small-en-v1.5"
CATEGORY_BY_FILE = {
    "hr_leave_policy.txt": "HR",
    "it_device_support.txt": "IT",
    "benefits_and_wellbeing.txt": "Benefits",
    "finance_expense_reimbursement.txt": "Finance",
    "people_onboarding.txt": "Onboarding",
    "security_data_protection.txt": "Security",
}

DATA_DIR = Path("/content/northstar_knowledge_base")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Ready to receive:", len(CATEGORY_BY_FILE), "knowledge documents")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
Ready to receive: 6 knowledge documents


## Substitute dataset disclosure — 2 November 2026

All 12 required functions were implemented with AI assistance. The original integration assertions and seven comparison cases are preserved.

**This version uses six AI-authored fictional substitute policy files, explicitly authorized by the student because the instructor ZIP was unavailable. These are NOT the instructor-supplied Northstar policies.** The setup cell reproducibly creates the documents and `Northstar_Substitute_Employee_Knowledge_Base.zip`, including a provenance notice. All reported results apply only to that substitute corpus.

The original assignment requests the supplied dataset; evaluator acceptance of this substitution is unconfirmed. This notebook must not be represented as having used the original dataset.

The optional DeepSeek extension is unused; no API key is needed. Execution and observations will be recorded below from actual runs.


In [2]:
# USER-AUTHORIZED SUBSTITUTE DATA. These are NOT the instructor's original policies.
import json, hashlib
SUBSTITUTE_FILES = {
  "hr_leave_policy.txt": "SUBSTITUTE TRAINING DATA — AI-authored, not the instructor-supplied dataset.\nHR: Time away and family emergencies. In this fictional organization, full-time employees receive 24 days of annual paid leave per calendar year. Submit ordinary vacation requests through the People Portal at least five working days ahead; the manager approves staffing coverage. An urgent family matter can qualify for up to three days of emergency compassionate leave. Notify the manager and People Operations as soon as reasonably possible; an emergency does not require advance booking. Sick leave is separate from annual vacation. Record absences accurately, avoid sharing unnecessary medical details, and contact People Operations for confidential guidance. Unused annual leave may carry forward up to five days with manager approval. These invented rules apply only to this substitute exercise.",
  "it_device_support.txt": "SUBSTITUTE TRAINING DATA — AI-authored, not the instructor-supplied dataset.\nIT: Device incidents and account access. If an employee laptop is broken, damaged, or stops working, open a Service Desk ticket with the asset tag and a short description of the symptoms. The IT hardware team diagnoses the fault and arranges a loan computer or replacement device when needed. Do not buy a replacement personally or send equipment to an unapproved repair shop. For a forgotten account password, use the corporate self-service password reset page; if locked out, contact the Service Desk to verify identity. Never send passwords or authentication codes in a support ticket. Include your contact number only in the internal approved ticketing system. Report lost equipment promptly so IT can protect work data and arrange recovery. These are invented training rules.",
  "benefits_and_wellbeing.txt": "SUBSTITUTE TRAINING DATA — AI-authored, not the instructor-supplied dataset.\nBenefits: Health coverage and wellbeing. Fictional employees can view company medical insurance, dependent eligibility, coverage summaries, and insurer contact details in the Benefits Portal. For questions about health cover, contact the Benefits team; the People Portal links to the current enrollment guide. Eligible staff may enroll a spouse and dependent children during the annual enrollment window or after a qualifying family event. Policy exclusions and claim decisions belong to the insurer; this exercise does not promise payment for a particular treatment. The wellbeing program offers confidential counseling appointments through an approved external provider. Managers receive no clinical notes from those sessions. Report incorrect enrollment details privately to Benefits and never attach medical records to a public discussion thread. These invented provisions describe only the substitute dataset.",
  "finance_expense_reimbursement.txt": "SUBSTITUTE TRAINING DATA — AI-authored, not the instructor-supplied dataset.\nFinance: Business travel expense reimbursement. Employees may reclaim reasonable taxi or cab fares incurred to attend a client meeting when the travel has a documented business purpose and manager approval. Upload the receipt, travel date, client or project code, and purpose in the Expense Portal within 15 calendar days of the journey. Finance reviews the claim and pays approved amounts in the next reimbursement cycle. Personal rides and ordinary home-to-office commuting are not eligible. For a missing receipt, request guidance from Finance rather than inventing supporting evidence. Foreign currency claims need the transaction currency and the exchange rate shown on the payment statement. Avoid publishing customer identities or payment-card numbers in shared examples. These financial procedures and deadlines are fictional substitute training content.",
  "people_onboarding.txt": "SUBSTITUTE TRAINING DATA — AI-authored, not the instructor-supplied dataset.\nOnboarding: Preparing for the first working day. Before joining, complete the employee information form, return the signed employment documents through the secure People Portal, and finish the assigned preboarding checklist. Confirm your starting time, meeting location or remote orientation link, and equipment delivery details with the onboarding coordinator. Upload requested identity documents only through the designated secure channel. On the first day, attend orientation and meet your manager to discuss responsibilities and the first-week plan. IT provides instructions for activating corporate accounts after identity checks. Required security awareness training must be completed within the first working week. If a preboarding task cannot be completed before the start date, notify the coordinator instead of entering fabricated details. These steps are fictional and belong to the substitute practice corpus.",
  "security_data_protection.txt": "SUBSTITUTE TRAINING DATA — AI-authored, not the instructor-supplied dataset.\nSecurity: Suspicious links and data protection. If you accidentally click a suspicious link, stop interacting with the page, do not enter credentials, and report the incident immediately to the Security Response team using the incident hotline or approved reporting form. Tell the team when it happened and whether you downloaded a file or entered a password. Follow the response team's containment instructions; preserve relevant messages rather than forwarding dangerous links to coworkers. If credentials were entered, contact Security and IT for account protection and credential reset. Do not investigate the malicious site yourself. Sensitive employee and customer records must remain in approved access-controlled storage, with least-privilege sharing. Report suspected accidental disclosure promptly. This invented security guidance is solely for the substitute training exercise."
}
DATA_DIR = Path('/content/northstar_substitute_knowledge_base')
DATA_DIR.mkdir(parents=True, exist_ok=True)
for name, text in SUBSTITUTE_FILES.items():
    (DATA_DIR / name).write_text(text, encoding='utf-8')
SUBSTITUTE_ZIP = Path('/content/Northstar_Substitute_Employee_Knowledge_Base.zip')
with zipfile.ZipFile(SUBSTITUTE_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in CATEGORY_BY_FILE:
        archive.write(DATA_DIR / name, arcname=name)
    archive.writestr('SUBSTITUTE_DATA_NOTICE.txt', 'AI-authored fictional substitute data, created with student authorization on 2 November 2026. Not the instructor-supplied dataset. Results apply only to this substitute corpus. Evaluator acceptance is unconfirmed.')
print('SUBSTITUTE DATA — not instructor-supplied:', len(SUBSTITUTE_FILES), 'files')
print('Archive:', SUBSTITUTE_ZIP.name)
print('SHA256:', hashlib.sha256(SUBSTITUTE_ZIP.read_bytes()).hexdigest())


SUBSTITUTE DATA — not instructor-supplied: 6 files
Archive: Northstar_Substitute_Employee_Knowledge_Base.zip
SHA256: 8a401fdcece02daad60587a5e2424cd44df0717e6164170f9bd0ecd5d5fb4fed


### Step 1. Read the employee knowledge documents

**Your task:** Complete `load_knowledge_files(data_dir)`.

- Read the six `.txt` files in the order of `CATEGORY_BY_FILE` (Python dictionaries preserve insertion order).
- For each file, load its UTF-8 text and trim surrounding whitespace.
- Create a dictionary with **four keys**: `id`, `source`, `category`, and `text`.
- Assign stable integer IDs starting at `1001` (`1001`, `1002`, …).
- Return a list containing one dictionary per non-empty document.

**Expected output:** Six document dictionaries, for example `{'id': 1001, 'source': 'hr_leave_policy.txt', 'category': 'HR', 'text': '...'}`.

**Hint:** Use `Path.read_text(encoding='utf-8')`, `enumerate(CATEGORY_BY_FILE, start=1001)`, and `CATEGORY_BY_FILE[filename]`.

**Purpose of Code:** Turn raw policy files into structured retrieval units with stable IDs and provenance metadata. Later search functions will use these fields to show which document supplied each result.

In [3]:
def load_knowledge_files(data_dir: Path) -> list[dict]:
    records = []
    for record_id, filename in enumerate(CATEGORY_BY_FILE, start=1001):
        text = (Path(data_dir) / filename).read_text(encoding="utf-8").strip()
        if text:
            records.append(dict(id=record_id, source=filename, category=CATEGORY_BY_FILE[filename], text=text))
    return records

## Part B — Context Engineering and a TF-IDF Baseline
### Step 2. Build a TF-IDF representation

**Your task:** Complete `build_tfidf_index(units)`.

- Extract document text from the structured records.
- Create a `TfidfVectorizer(stop_words='english')`.
- Fit it on the documents and obtain the document-term matrix.
- Return `(vectorizer, tfidf_matrix)`.

**Expected output:** A fitted vectorizer and a matrix with one row per document.

**Hint:** Recall the lexical retrieval baseline from *Introduction to Context Engineering*.

**Purpose of Code:** Create a simple retrieval baseline that scores documents using overlapping vocabulary before introducing dense semantic embeddings.

In [4]:
def build_tfidf_index(units: list[dict]):
    vectorizer = TfidfVectorizer(stop_words="english")
    return vectorizer, vectorizer.fit_transform([u["text"] for u in units])

### Step 3. Retrieve documents using TF-IDF and cosine similarity

**Your task:** Complete `tfidf_search(question, units, vectorizer, tfidf_matrix, top_k=3)`.

- Transform the question using the **same fitted vectorizer**.
- Compute cosine similarity between the question vector and document matrix.
- Rank records from highest score to lowest; return up to `top_k` matches.
- Each returned dictionary must include `id`, `source`, `category`, `text`, and a floating-point `score`.
- **Ignore zero-score documents.** If the question has no lexical match, return `[]`.

**Example output:** `[{'source': 'it_device_support.txt', ..., 'score': 0.29}, ...]` (score shown only as an illustration).

**Hint:** Use `cosine_similarity(q_vector, tfidf_matrix)[0]` and `np.argsort(scores)[::-1]`.

**Purpose of Code:** Rank the policy documents by shared terms and expose when keyword-based retrieval has no signal. This gives a baseline to compare with FAISS and ChromaDB.

In [5]:
def tfidf_search(question, units, vectorizer, tfidf_matrix, top_k=3):
    if not units or top_k <= 0:
        return []
    scores = cosine_similarity(vectorizer.transform([question]), tfidf_matrix)[0]
    ranked = sorted(range(len(units)), key=lambda i: (-float(scores[i]), units[i]["id"]))
    return [dict(units[i], score=float(scores[i])) for i in ranked if scores[i] > 0][:top_k]

### Step 4. Assemble a readable context block

**Your task:** Complete `format_context(results, max_chars=1600)`.

- Turn a list of retrieved matches into a single readable text block.
- Prefix each passage with a **source label** such as `[Source: it_device_support.txt | IT]`.
- Include the matching document text; stay within the total `max_chars` character budget.
- If no passages are provided, return **exactly** `Insufficient context.`.

**Expected output:** A source-aware context string ready to be placed in an LLM prompt (even though LLM generation is optional here).

**Hint:** Use `result['source']`, `result['category']`, `result['text']`, and `len()` to keep the output within the budget.

**Purpose of Code:** Build relevant, traceable, and length-controlled context. This demonstrates context engineering beyond retrieving a list of passages.

In [6]:
def format_context(results: list[dict], max_chars: int = 1600) -> str:
    if not results:
        return "Insufficient context."
    if max_chars < 0:
        raise ValueError("max_chars must be non-negative")
    blocks = [f"[Source: {r['source']} | {r['category']}]\n{r['text']}" for r in results]
    return "\n\n".join(blocks)[:max_chars]

## Part C — Embeddings and FAISS Semantic Search
### Step 5. Generate dense document embeddings

**Your task:** Complete `embed_documents(units, model)`.

- Encode all document texts with the supplied `SentenceTransformer` model.
- Use `convert_to_numpy=True` and `normalize_embeddings=True`.
- Convert the result to a contiguous, two-dimensional NumPy array of `float32` values.
- Return the embedding matrix; do not create a vector index in this step.

**Expected output:** A matrix with six rows and one embedding vector per document.

**Hint:** See the `model.encode(...)` usage in *Introduction to FAISS*.

**Purpose of Code:** Convert document meaning into dense numerical vectors so that semantically related questions can match policies even when their wording is different.

In [7]:
def embed_documents(units, model):
    vectors = model.encode([u["text"] for u in units], convert_to_numpy=True, normalize_embeddings=True)
    return np.ascontiguousarray(vectors, dtype=np.float32)

### Step 6. Build a FAISS vector index with stable IDs

**Your task:** Complete `build_faiss_index(vectors, units)`.

- Create an `IndexFlatIP` using the embedding dimension.
- Wrap it in `faiss.IndexIDMap`.
- Add the normalized vectors with their stable integer `id` values using `add_with_ids`.
- Create a lookup dictionary mapping `id -> original document record`.
- Return `(index, units_by_id)`.

**Expected output:** A FAISS index containing six records and a dictionary that restores source text/metadata from search IDs.

**Hint:** Use `vectors.shape[1]`, `np.asarray(..., dtype=np.int64)`, and `faiss.IndexIDMap(faiss.IndexFlatIP(dimension))`.

**Purpose of Code:** Store normalized semantic vectors in a searchable FAISS index and preserve stable IDs so returned matches can be connected to their original policy documents.

In [8]:
def build_faiss_index(vectors, units):
    if vectors.ndim != 2 or len(vectors) != len(units):
        raise ValueError("Each document must have one embedding")
    lookup = {int(u["id"]): dict(u) for u in units}
    if len(lookup) != len(units):
        raise ValueError("Document IDs must be unique")
    index = faiss.IndexIDMap(faiss.IndexFlatIP(vectors.shape[1]))
    index.add_with_ids(np.ascontiguousarray(vectors, dtype=np.float32), np.asarray([u["id"] for u in units], dtype=np.int64))
    return index, lookup

### Step 7. Search with FAISS

**Your task:** Complete `faiss_search(question, model, index, units_by_id, top_k=3)`.

- Encode and normalize the question using the **same embedding model**.
- Search the FAISS index for up to `top_k` nearest results.
- Convert valid returned IDs back into the original records.
- Return dictionaries with `id`, `source`, `category`, `text`, and a floating-point `score`.
- Skip FAISS's `-1` ID if encountered.

**Expected output:** Ranked semantic document matches with source labels.

**Hint:** `index.search(query_vector, k)` returns `(scores, ids)`; scores from normalized `IndexFlatIP` are cosine-style similarities.

**Purpose of Code:** Retrieve policy documents based on semantic similarity, including questions that do not share exact vocabulary with their relevant documents.

In [9]:
def faiss_search(question, model, index, units_by_id, top_k=3):
    if top_k <= 0 or index.ntotal == 0:
        return []
    query = np.ascontiguousarray(model.encode([question], convert_to_numpy=True, normalize_embeddings=True), dtype=np.float32)
    scores, ids = index.search(query, min(top_k, index.ntotal))
    return [dict(units_by_id[int(i)], score=float(s)) for s, i in zip(scores[0], ids[0]) if i != -1 and int(i) in units_by_id]

## Part D — LlamaIndex Documents and Nodes
### Step 8. Convert records into LlamaIndex Documents

**Your task:** Complete `create_llama_documents(units)`.

- Construct one `Document(text=..., metadata=...)` for each structured policy record.
- Each document must retain `source`, `category`, and `record_id` metadata.
- Return the list of LlamaIndex documents.

**Expected output:** Six `Document` objects with source-aware metadata.

**Hint:** This matches the document abstraction used in *Advanced Vector Databases with ChromaDB & LlamaIndex*.

**Purpose of Code:** Represent the policies as LlamaIndex `Document` objects while preserving the provenance needed for filtering, source display, and later updates.

In [10]:
def create_llama_documents(units):
    return [Document(text=u["text"], metadata={"source": u["source"], "category": u["category"], "record_id": int(u["id"])}) for u in units]

### Step 9. Build an ingestion pipeline and create Nodes

**Your task:** Complete `create_nodes(documents)`.

- Create `SentenceSplitter(chunk_size=180, chunk_overlap=20)`.
- Create an `IngestionPipeline(transformations=[splitter])`.
- Run the pipeline on `documents` and return the resulting nodes.
- Do not remove `source` or `category` metadata from the nodes.

**Expected output:** A non-empty list of smaller retrieval-ready Nodes.

**Hint:** `pipeline.run(documents=documents)` was demonstrated in Lesson 3.

**Purpose of Code:** Divide larger documents into LlamaIndex Nodes so ChromaDB can retrieve focused passages rather than always returning an entire policy file.

In [11]:
def create_nodes(documents):
    splitter = SentenceSplitter(chunk_size=180, chunk_overlap=20)
    return IngestionPipeline(transformations=[splitter]).run(documents=documents)

## Part E — ChromaDB Storage, Metadata Search, and Updates
### Step 10. Save LlamaIndex Nodes in persistent ChromaDB

**Your task:** Complete `upsert_nodes_to_chroma(nodes, model, collection)`.

- Get each Node's text via `node.get_content()` and keep its metadata.
- Generate one **normalized `float32` embedding** for each node using the same model as FAISS.
- Create deterministic IDs in input order: `node-0000`, `node-0001`, etc. (this fixed lab dataset makes the order stable).
- Use `collection.upsert(ids=..., documents=..., metadatas=..., embeddings=...)`.
- Return the number of Nodes upserted.

**Expected output:** The number of retrieval-ready records stored in the collection. Running the same dataset twice should **update**, not duplicate, records.

**Hint:** Chroma accepts embedding arrays as lists; use `vectors.tolist()`. Metadata must contain `source` and `category` from each Node.

**Purpose of Code:** Persist LlamaIndex Nodes together with their embeddings and metadata, allowing semantic queries, metadata filters, and record updates in ChromaDB.

In [12]:
def upsert_nodes_to_chroma(nodes, model, collection):
    if not nodes:
        return 0
    texts = [n.get_content() for n in nodes]
    vectors = np.ascontiguousarray(model.encode(texts, convert_to_numpy=True, normalize_embeddings=True), dtype=np.float32)
    collection.upsert(ids=[f"node-{i:04d}" for i in range(len(nodes))], documents=texts, metadatas=[dict(n.metadata) for n in nodes], embeddings=vectors.tolist())
    return len(nodes)

### Step 11. Search ChromaDB with an optional metadata filter

**Your task:** Complete `chroma_search(question, model, collection, top_k=3, category=None)`.

- Encode the question with the **same normalized embedding model**.
- Use `collection.query` with the query embedding and `n_results`.
- When `category` is supplied (for example, `'IT'`), apply `where={'category': category}`.
- Extract matching `documents`, `metadatas`, `ids`, and `distances`.
- Return dictionaries with `id`, `source`, `category`, `text`, and `distance`.
- If the collection is empty, return `[]`.

**Important:** Chroma returns **distances** (smaller is better), whereas normalized FAISS `IndexFlatIP` returns **similarity scores** (larger is better). Do not compare those raw values directly.

**Purpose of Code:** Retrieve semantically relevant LlamaIndex Nodes from persistent ChromaDB, optionally narrowing search to a particular policy department before ranking.

In [13]:
def chroma_search(question, model, collection, top_k=3, category=None):
    count = collection.count()
    if not count or top_k <= 0:
        return []
    query = np.ascontiguousarray(model.encode([question], convert_to_numpy=True, normalize_embeddings=True), dtype=np.float32)
    options = {"where": {"category": category}} if category is not None else {}
    result = collection.query(query_embeddings=query.tolist(), n_results=min(top_k, count), include=["documents", "metadatas", "distances"], **options)
    return [dict(id=i, source=m["source"], category=m["category"], text=t, distance=float(d)) for i, t, m, d in zip(result["ids"][0], result["documents"][0], result["metadatas"][0], result["distances"][0])]

### Step 12. Update ChromaDB record metadata

**Your task:** Complete `update_record_status(collection, record_id, status)`.

- Read the Chroma record identified by `record_id` using `collection.get`.
- If no such ID is present, raise `ValueError`.
- Keep its existing metadata and add/update `review_status` with the provided `status`.
- Call `collection.update` to save the new metadata.
- Read the updated record again and return its metadata dictionary.

**Expected output:** Metadata that includes the original `source`, `category`, and the requested `review_status`.

**Hint:** Use `.get(ids=[...], include=['metadatas'])` and `.update(ids=[...], metadatas=[...])`.

**Purpose of Code:** Demonstrate how a managed vector collection supports record updates without discarding provenance or recreating the entire database.

In [14]:
def update_record_status(collection, record_id, status):
    result = collection.get(ids=[record_id], include=["metadatas"])
    if not result["ids"]:
        raise ValueError(f"Unknown record ID: {record_id}")
    metadata = dict(result["metadatas"][0] or {})
    metadata["review_status"] = status
    collection.update(ids=[record_id], metadatas=[metadata])
    return dict(collection.get(ids=[record_id], include=["metadatas"])["metadatas"][0])

## Part F — Connect the Components and Test Retrieval
### Step 13. Run the integration checks

**Your task:** Run the provided cell once all 12 functions are implemented. Resolve any errors before continuing.

The cell checks document loading, TF-IDF, normalized embeddings, FAISS indexing, LlamaIndex Nodes, Chroma persistence, department filtering, context formatting, and metadata updates. These are **structural checks**, not a guarantee that every search result is correct.

**Purpose of Code:** Verify that the individual student-written functions work together as one retrieval pipeline and preserve the document-to-metadata links throughout the workflow.

In [15]:
# PROVIDED TEST HARNESS — Execute only after completing Steps 1–12.
units = load_knowledge_files(DATA_DIR)
assert len(units) == 6, "Expected exactly six knowledge files."
assert len({u["id"] for u in units}) == 6, "Every document needs a unique ID."
assert all({"id", "source", "category", "text"} <= set(u) for u in units)
assert all(isinstance(u["text"], str) and u["text"].strip() for u in units)
print("Document loading:", len(units), "records")

vectorizer, tfidf_matrix = build_tfidf_index(units)
assert tfidf_matrix.shape[0] == len(units)
lexical = tfidf_search("How do I report a broken laptop?", units, vectorizer, tfidf_matrix)
assert isinstance(lexical, list)
print("TF-IDF first source:", lexical[0]["source"] if lexical else "No match")

model = SentenceTransformer(MODEL_NAME)
vectors = embed_documents(units, model)
assert isinstance(vectors, np.ndarray) and vectors.ndim == 2
assert vectors.shape[0] == len(units) and vectors.dtype == np.float32
assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-3)
index, units_by_id = build_faiss_index(vectors, units)
assert index.ntotal == len(units)
semantic = faiss_search("My work computer stopped working", model, index, units_by_id)
assert isinstance(semantic, list) and semantic
print("FAISS first source:", semantic[0]["source"])

llama_documents = create_llama_documents(units)
assert len(llama_documents) == len(units)
assert all("source" in doc.metadata and "category" in doc.metadata for doc in llama_documents)
nodes = create_nodes(llama_documents)
assert len(nodes) >= len(llama_documents)
assert all("source" in node.metadata and "category" in node.metadata for node in nodes)
print("LlamaIndex nodes:", len(nodes))

chroma_client = chromadb.PersistentClient(path="/content/northstar_chroma_data")
collection = chroma_client.get_or_create_collection(name="northstar_employee_knowledge")
stored_count = upsert_nodes_to_chroma(nodes, model, collection)
assert stored_count == len(nodes)
assert collection.count() >= len(nodes)
print("Chroma stored records:", collection.count())

filtered = chroma_search("My laptop is damaged", model, collection, top_k=3, category="IT")
assert filtered and all(r["category"] == "IT" for r in filtered)
print("Filtered result source:", filtered[0]["source"])

sample_context = format_context(semantic[:2], max_chars=600)
assert isinstance(sample_context, str) and len(sample_context) <= 600
assert "Source:" in sample_context
assert format_context([]) == "Insufficient context."
print("Context preview:\n", sample_context[:350])

sample_id = collection.get(limit=1)["ids"][0]
updated = update_record_status(collection, sample_id, "approved")
assert updated["review_status"] == "approved"
assert "source" in updated and "category" in updated
print("Updated record metadata:", updated)
print("All structural checks passed.")

Document loading: 6 records
TF-IDF first source: it_device_support.txt


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS first source: it_device_support.txt
LlamaIndex nodes: 7
Chroma stored records: 7
Filtered result source: it_device_support.txt
Context preview:
 [Source: it_device_support.txt | IT]
SUBSTITUTE TRAINING DATA — AI-authored, not the instructor-supplied dataset.
IT: Device incidents and account access. If an employee laptop is broken, damaged, or stops working, open a Service Desk ticket with the asset tag and a short description of the symptoms. The IT hardware team diagnoses the fault and arr
Updated record metadata: {'review_status': 'approved', 'record_id': 1001, 'category': 'HR', 'source': 'hr_leave_policy.txt'}
All structural checks passed.


### Step 14. Compare lexical and semantic retrieval on fixed questions

**Your task:** Run the provided comparison cell and inspect the results.

- For every question, inspect the first returned **source file**, not just the score.
- The `Expected source` column is an instructor-provided reference, **not** a guarantee that an algorithm will rank it first.
- Examine at least **two questions using different wording** from the documents.
- For the out-of-scope question, describe why a high-ranked result can still be irrelevant.
- A Chroma result is a **Node**, whereas TF-IDF and FAISS search the **whole text file**. Compare source relevance carefully; these are not identical retrieval granularities.

**Purpose of Code:** Evaluate whether retrieval returns the relevant source documents and understand how different representations and retrieval units affect search behavior.

In [16]:
TEST_CASES = [
    ("My work computer stopped working. Who arranges a replacement?", "it_device_support.txt"),
    ("Can I take time off for an urgent family matter?", "hr_leave_policy.txt"),
    ("Can I get money back for a cab to a client meeting?", "finance_expense_reimbursement.txt"),
    ("Where do I learn about the company health cover?", "benefits_and_wellbeing.txt"),
    ("What should I finish before my first working day?", "people_onboarding.txt"),
    ("I accidentally clicked a suspicious link. What now?", "security_data_protection.txt"),
    ("What does the cafeteria serve next Tuesday?", None),  # Out of scope
]

rows = []
for question, expected in TEST_CASES:
    lexical_hits = tfidf_search(question, units, vectorizer, tfidf_matrix, top_k=3)
    faiss_hits = faiss_search(question, model, index, units_by_id, top_k=3)
    chroma_hits = chroma_search(question, model, collection, top_k=3)
    first_source = lambda results: results[0]["source"] if results else "No match"
    rows.append({
        "Question": question,
        "Expected source": expected or "Not in documents",
        "TF-IDF top source": first_source(lexical_hits),
        "FAISS top source": first_source(faiss_hits),
        "Chroma top source": first_source(chroma_hits),
        "TF-IDF source hit": expected is not None and first_source(lexical_hits) == expected,
        "FAISS source hit": expected is not None and first_source(faiss_hits) == expected,
        "Chroma source hit": expected is not None and first_source(chroma_hits) == expected,
    })

comparison_df = pd.DataFrame(rows)
display(comparison_df)
known_questions = comparison_df[comparison_df["Expected source"] != "Not in documents"]
for column in ["TF-IDF source hit", "FAISS source hit", "Chroma source hit"]:
    print(f"{column}: {known_questions[column].sum()} / {len(known_questions)}")

print("\nIT-only Chroma results:")
for item in chroma_search("I need help with an account password", model, collection, category="IT"):
    print(item["source"], item["category"], item["text"][:100])

,Question,Expected source,TF-IDF top source,FAISS top source,Chroma top source,TF-IDF source hit,FAISS source hit,Chroma source hit
0,My work computer stopped working. Who arranges...,it_device_support.txt,it_device_support.txt,it_device_support.txt,it_device_support.txt,True,True,True
1,Can I take time off for an urgent family matter?,hr_leave_policy.txt,hr_leave_policy.txt,hr_leave_policy.txt,hr_leave_policy.txt,True,True,True
2,Can I get money back for a cab to a client mee...,finance_expense_reimbursement.txt,finance_expense_reimbursement.txt,finance_expense_reimbursement.txt,finance_expense_reimbursement.txt,True,True,True
3,Where do I learn about the company health cover?,benefits_and_wellbeing.txt,benefits_and_wellbeing.txt,benefits_and_wellbeing.txt,benefits_and_wellbeing.txt,True,True,True
4,What should I finish before my first working day?,people_onboarding.txt,people_onboarding.txt,people_onboarding.txt,people_onboarding.txt,True,True,True
5,I accidentally clicked a suspicious link. What...,security_data_protection.txt,security_data_protection.txt,security_data_protection.txt,security_data_protection.txt,True,True,True
6,What does the cafeteria serve next Tuesday?,Not in documents,benefits_and_wellbeing.txt,hr_leave_policy.txt,hr_leave_policy.txt,False,False,False


TF-IDF source hit: 6 / 6
FAISS source hit: 6 / 6
Chroma source hit: 6 / 6

IT-only Chroma results:
it_device_support.txt IT SUBSTITUTE TRAINING DATA — AI-authored, not the instructor-supplied dataset.
IT: Device incidents an


### Step 15. Write your observations (Markdown response)

Add a **new Markdown cell** after this one and answer in your own words:

1. For which two questions did TF-IDF and FAISS differ? Which retrieved source was more useful, and why?
2. How did the ChromaDB `category='IT'` filter change the results?
3. Why might comparing raw FAISS scores with raw Chroma distances be misleading?
4. Explain why the out-of-scope cafeteria question must not be treated as answered, even if the vector store retrieves something.
5. What are two ways that metadata and context limits improve the usefulness or safety of an LLM's final answer?

**Purpose of this step:** Interpret the search outputs rather than assuming that receiving a result means it is correct or sufficiently grounded.

In [17]:
# Extra paraphrases: preserve the original seven cases above unchanged.
PROBES = [
    ('My workstation is kaput; can someone supply a temporary machine?', 'it_device_support.txt'),
    ('Is there cover for bereavement-related absence?', 'hr_leave_policy.txt'),
]
probe_rows = []
for question, expected in PROBES:
    methods = {
        'TF-IDF': tfidf_search(question, units, vectorizer, tfidf_matrix),
        'FAISS': faiss_search(question, model, index, units_by_id),
        'Chroma': chroma_search(question, model, collection),
    }
    probe_rows.append({'Question': question, 'Expected': expected,
        **{name: hits[0]['source'] if hits else 'No match' for name, hits in methods.items()}})
display(pd.DataFrame(probe_rows))
# Check rerun behavior without altering the source corpus or the reference labels.
before = collection.count()
upsert_nodes_to_chroma(nodes, model, collection)
assert collection.count() == before == len(nodes)
assert len(set(collection.get()['ids'])) == len(nodes)
assert chromadb.PersistentClient(path='/content/northstar_chroma_data').get_collection('northstar_employee_knowledge').count() == before
assert tfidf_search('zzzxxyyqq', units, vectorizer, tfidf_matrix) == []
assert len(format_context(semantic, max_chars=80)) <= 80
try:
    update_record_status(collection, 'missing-record', 'approved')
except ValueError:
    print('Missing record correctly rejected.')
else:
    raise AssertionError('Missing record must raise ValueError')
updated = update_record_status(collection, sample_id, 'approved')
print('Idempotent upsert, persistent reopen, zero lexical signal, context budget, and missing-ID checks passed.')
print('Records:', collection.count(), '| review status:', updated['review_status'])


,Question,Expected,TF-IDF,FAISS,Chroma
0,My workstation is kaput; can someone supply a ...,it_device_support.txt,No match,it_device_support.txt,it_device_support.txt
1,Is there cover for bereavement-related absence?,hr_leave_policy.txt,benefits_and_wellbeing.txt,benefits_and_wellbeing.txt,benefits_and_wellbeing.txt


Missing record correctly rejected.
Idempotent upsert, persistent reopen, zero lexical signal, context budget, and missing-ID checks passed.
Records: 7 | review status: approved


## Step 15 — Observations from the actual substitute-data run

**Provenance:** all observations below use the six AI-authored substitute policies, not the unavailable instructor ZIP. The corpus was written with the visible lab topics and test cases in mind; this is a development demonstration, not an independent held-out benchmark or an official lab grade.

### Results
- Loaded six unique document records (IDs 1001–1006). LlamaIndex produced seven Nodes; Chroma persisted seven records.
- The unchanged integration harness printed **All structural checks passed.**
- On the six in-scope reference questions, TF-IDF, FAISS and Chroma each retrieved the expected source first: **6/6 for each method**. These are source-level hits, not an assessment of generated answers.
- Re-upserting the same Nodes kept the count at seven; a second persistent client saw the same seven records. Zero-signal lexical search returned no results, context stayed within its character budget, and updating a nonexistent ID raised ValueError. Metadata review_status was updated to approved without losing source/category.

### 1. Retrieval differences and paraphrases
The six in-scope reference questions did **not** produce two TF-IDF/FAISS disagreements; all methods agreed. I did not change the reference questions or invent disagreements. Two supplementary probes show the limits more clearly:

| Extra query | TF-IDF first source | FAISS / Chroma first source | Interpretation |
|---|---|---|---|
| My workstation is kaput; can someone supply a temporary machine? | No match | it_device_support.txt | Dense retrieval recognized a device problem despite missing useful lexical overlap. The IT policy discusses diagnosis and loan/replacement devices. |
| Is there cover for bereavement-related absence? | benefits_and_wellbeing.txt | benefits_and_wellbeing.txt | All methods ranked Benefits first. The word cover is ambiguous; HR is the manually intended category for compassionate leave. Semantic similarity did not fix this ambiguity. Clarify the request or search multiple sources. |

### 2. Department filtering
With category='IT', all returned Chroma Nodes came from it_device_support.txt. The filter restricts the candidate set before ranking and excludes other departments; it does not prove that a passage answers the question. The filter is a demonstration category, not a complete authorization system.

### 3. Similarity versus distance
FAISS uses normalized IndexFlatIP vectors and reports larger-is-better cosine-style similarity. The default Chroma collection reports smaller-is-better L2 distances. Raw numbers should not be compared directly. Chroma searches Nodes while FAISS searches entire documents, so even comparable metrics would not make these identical retrieval tasks.

### 4. Out-of-scope cafeteria question
For 'What does the cafeteria serve next Tuesday?', TF-IDF returned Benefits and FAISS/Chroma returned HR. None of the substitute policies contains a cafeteria menu. The returned neighbors therefore provide **insufficient context**, not a supported answer. Retrieval is not answerability detection; lexical overlap and a high-ranked vector result can both mislead. The optional LLM extension was not run.

### 5. Metadata and context limits
Source/category metadata preserves provenance and supports focused filtering; real access rules must be enforced by trusted application logic. Character budgets limit unnecessary input and cost, but truncating a passage can remove qualifications. Preserve source labels, inspect completeness, and avoid interpreting an arbitrarily clipped passage as the entire policy.

### Remaining practical limitations
This is a small, purpose-written substitute corpus. Node IDs are deterministic by input order only and need a stronger source/version identity strategy for changing corpora. No automatic answerability threshold or reliable refusal mechanism is implemented. Evaluator acceptance of substitute data remains unconfirmed.


## Optional Extension — Generate a Grounded Answer with DeepSeek

This is **not required for core marks**. If you have time and already know how to use an LLM API from GenAI Foundations, you may build a short answer generator.

**Your task:** Complete the optional function below using the API pattern from the previous module.

- Retrieve at most two Chroma matches and format them with `format_context`.
- Use a system instruction telling the LLM to answer **only from the provided context**, identify the source, and say `Insufficient context.` when evidence is missing.
- Read `DEEPSEEK_API_KEY` securely from Colab Secrets; never print or commit it.
- Treat retrieved results as **possible evidence**, not proof of an answer. Test the out-of-scope question explicitly.

**Note:** Vector search always tries to find near neighbors, even for an unrelated question. A prompt alone cannot guarantee reliable refusal; discuss how stronger relevance checks could help.

**Purpose of Code:** Connect your improved retrieval pipeline to the LLM generation techniques from GenAI Foundations without making API access a requirement of this module.

In [ ]:
# OPTIONAL: requires a DeepSeek API key and additional request setup from the earlier chapter.
def generate_grounded_answer(question: str, retrieved_results: list[dict]) -> str:
    """Return an LLM answer using only source-labelled retrieved context."""
    # TODO 1: Format at most two retrieved passages using format_context.
    # TODO 2: Prepare a grounded system/user prompt and source instructions.
    # TODO 3: Call the LLM API using your earlier chapter's secure setup.
    # TODO 4: Return the generated text (or an insufficient-context response).
    pass

## Evaluation Rubric

| Component | Marks |
|---|---:|
| Load documents with stable IDs and metadata | 10 |
| TF-IDF ranking and context formatting | 20 |
| Normalized dense embeddings and FAISS retrieval | 25 |
| LlamaIndex Documents and Nodes | 15 |
| ChromaDB storage, filtering, and metadata update | 20 |
| Testing, interpretation, and notebook clarity | 10 |
| **Total** | **100** |

The optional LLM extension is for extra practice, not part of the required 100 marks. Evaluate *correctness and meaningful interpretation*, not whether every retrieval method always returns the expected source first.

## Submission Checklist

- [ ] I implemented **all 12 required functions** in Steps 1–12.
- [ ] I used the **six supplied Northstar policy files** and kept their source/category metadata.
- [ ] My Colab notebook runs the integration checks and displays the retrieval comparison table.
- [ ] I tested the ChromaDB category filter and updated record metadata.
- [ ] I explained retrieval differences and the out-of-scope question in a Markdown cell.
- [ ] I uploaded the **executed notebook** and the supplied data ZIP/files to GitHub.
- [ ] I did not include API keys, ChromaDB generated folders, or model-cache directories in my repository.

**Reminder:** Leave all generated outputs visible in your submitted notebook.  